<a href="https://colab.research.google.com/github/camilla32/Projeto-de-cria-o-de-pepiline-de-ETL/blob/main/Pepiline_de_ETL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install pandas sqlalchemy

In [4]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

# =========================
# 1. EXTRACT (gerando dados)
# =========================
def extract_data():
    print("Extraindo dados (gerando dados sintéticos)...")

    np.random.seed(42)

    df = pd.DataFrame({
        'id_pedido': range(1, 31),
        'cliente': [f'Cliente_{i}' for i in range(1, 31)],
        'valor': np.random.randint(100, 5000, 30),
        'categoria': np.random.choice(['Eletrônicos', 'Roupas', 'Alimentos'], 30),
        'data_pedido': pd.date_range(start='2024-01-01', periods=30)
    })

    return df


# =========================
# 2. TRANSFORM
# =========================
def transform_data(df):
    print("Transformando dados...")

    # Padronizar colunas
    df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')

    # Converter tipos
    df['valor'] = pd.to_numeric(df['valor'], errors='coerce')
    df['data_pedido'] = pd.to_datetime(df['data_pedido'])

    # Criar coluna de classificação de valor
    df['categoria_valor'] = df['valor'].apply(
        lambda x: 'Alto' if x > 3000 else 'Médio' if x > 1000 else 'Baixo'
    )

    # Criar coluna de mês
    df['mes'] = df['data_pedido'].dt.month

    return df


# =========================
# 3. LOAD
# =========================
def load_data(df):
    print("Carregando dados...")

    # Salvar CSV
    df.to_csv('/content/dados_tratados.csv', index=False)

    # Salvar em banco SQLite
    engine = create_engine('sqlite:///etl_compras.db')
    df.to_sql('compras', engine, if_exists='replace', index=False)

    print("Dados salvos em CSV e banco SQLite!")


# =========================
# PIPELINE
# =========================
def etl_pipeline():
    df = extract_data()
    df = transform_data(df)
    load_data(df)

    return df


# =========================
# EXECUÇÃO
# =========================
df_final = etl_pipeline()

df_final.head()

Extraindo dados (gerando dados sintéticos)...
Transformando dados...
Carregando dados...
Dados salvos em CSV e banco SQLite!


,id_pedido,cliente,valor,categoria,data_pedido,categoria_valor,mes
0,1,Cliente_1,960,Alimentos,2024-01-01,Baixo,1
1,2,Cliente_2,3872,Eletrônicos,2024-01-02,Alto,1
2,3,Cliente_3,3192,Alimentos,2024-01-03,Alto,1
3,4,Cliente_4,566,Eletrônicos,2024-01-04,Baixo,1
4,5,Cliente_5,4526,Alimentos,2024-01-05,Alto,1
